# Computer Organisation & Architecture — CO1

**2310221L.CO.1** — *Explicate the architecture and instruction set of the 80386
microprocessor.* **[L2]**

This notebook covers **CO1 only**, in depth. The other outcomes are in
[`subjects/02-computer-organisation.md`](subjects/02-computer-organisation.md).

---

## What "architecture" means

Architecture is the answer to two questions:

1. **What parts exist**, and what is each one for?
2. **What can a program ask the machine to do** — the instruction set?

Everything below answers one of those two.

![the datapath](diagrams/coa-datapath.png)

---

## Part 1 · The parts, and what each is for

### The word — how a value is represented

Before anything else, a machine has to decide how wide a number is. Ours is 16 bits. That
decision determines the largest value a register can hold (65,535), what happens when you
exceed it, and how much memory each cell takes.

**What this does.** Adds two machine words. The cast is what makes the result wrap around
at 16 bits rather than growing — which is why `0 - 1` gives 65,535 and not −1.

```cpp
Word operator+(const Word& o) const { return Word(static_cast<u16>(value_ + o.value_)); }
```
<sub>src/core/Word.h:44</sub>

### The registers — the machine's working space

Registers are the small, fast store the processor works from. Ours:

| Register | What it is for |
|---|---|
| R0 – R7 | general purpose — the program uses these freely |
| PC | program counter — which instruction comes next |
| IR | instruction register — the one being carried out now |
| SP | stack pointer — how deep the call stack is |
| MAR | memory address register — which address is being accessed |
| MDR | memory data register — the value going to or coming from memory |

The last four are not there for the programmer. They exist because a processor cannot
reach memory in one motion: it must first put an address somewhere the memory can see it
(MAR), then collect what comes back (MDR). Naming those registers is part of explaining
the architecture.

**What this does.** Reads a register and quietly records that it was read during this
cycle. That record is what lets the display highlight exactly which registers took part in
the current instruction.

```cpp
Word read()  { readThisCycle_ = true;  return value_; }
void write(Word v) { value_ = v; wroteThisCycle_ = true; }
```
<sub>src/core/RegisterFile.h:36</sub>

### The ALU — where arithmetic happens

The arithmetic and logic unit performs one operation on one or two values and reports what
happened.

![the ALU and its flags](diagrams/coa-alu-flags.png)

**What this does.** The ALU's whole interface. Give it an operation and two values, and it
returns the result together with the flags. It keeps nothing between calls, which is why
it can be tested entirely on its own.

```cpp
AluResult execute(AluOp op, Word a, Word b);
```
<sub>src/core/ALU.h:82</sub>

### The flags — how a machine decides

This is the part most worth explaining, because it is where a calculator becomes a
computer.

An ALU that only produced answers could compute but never *choose*. The flags are four
yes/no facts recorded alongside every result:

| Flag | Set when |
|---|---|
| **Z** zero | the result came out as 0 |
| **C** carry | the answer did not fit — a bit fell off the top |
| **V** overflow | the signed answer was too large to represent |
| **N** negative | the top bit is set, so read as signed it is negative |

`CMP R3, R2` subtracts, throws the answer away, and keeps only the flags. `JNZ` then looks
at the zero flag and decides whether to jump. **That pair is what a loop is made of** —
without flags, a program could only run straight through from top to bottom.

### Memory — larger, slower, further away

Registers are few and immediate; memory is large and reached through an address.

**What this does.** Holds memory as address-and-value pairs rather than one enormous
array, so only cells actually written take up space. Anything never written reads as zero.

```cpp
class Memory {
    ds::HashMap<unsigned int, u16> cells_;   // address -> value, sparse
};
```
<sub>src/core/Memory.h:22</sub>

### The bus — how the parts are connected

All of them share one data bus. Only one thing may drive it at a time, which is exactly
why a control unit is needed to say whose turn it is. In the diagram at the top, the bus
is the long horizontal line, and each block drops onto it.

---

## Part 2 · The instruction set

An instruction set is the complete list of things a program is allowed to ask for. Ours
has fourteen.

| Instruction | What it does |
|---|---|
| `LOAD Rd, #n` | put a fixed number into a register |
| `LOADM Rd, [addr]` | copy a value out of memory into a register |
| `STORE Rs, [addr]` | copy a register into memory |
| `MOV Rd, Rs` | copy one register into another |
| `ADD / SUB Rd, Rs` | arithmetic, result into Rd |
| `AND / OR / XOR Rd, Rs` | bitwise logic |
| `CMP Rd, Rs` | compare — sets the flags, writes nothing |
| `INC / DEC Rd` | add or subtract one |
| `JMP addr` | jump, always |
| `JZ / JNZ addr` | jump only if the zero flag is, or is not, set |
| `OUT Rs` | print a register |
| `HLT` | stop |

### Addressing modes — how an instruction names its operand

Three ways, which is enough for every instruction we need:

| Mode | Example | The operand is |
|---|---|---|
| immediate | `LOAD R0, #5` | written into the instruction itself |
| register | `ADD R0, R1` | in a register, named by number |
| direct | `STORE R0, [0x40]` | in memory, at an address given in the instruction |

### The instruction as an object

Every instruction knows two things about itself: how to carry itself out, and which
control lines it needs while doing so.

**What this does.** The base every instruction is built from. These two lines are the
whole contract of the instruction set.

```cpp
class Instruction {
public:
    virtual void           execute(CPU& cpu) = 0;
    virtual ControlSignals signals() const   = 0;
};
```
<sub>src/core/Instruction.h:65</sub>

---

## Part 3 · Carrying out one instruction

Explaining the parts and the instruction set is not enough on its own. CO1 is about
*explicating* the machine, and the thing that ties the parts to the instructions is the
cycle.

![the instruction cycle](diagrams/coa-instruction-cycle.png)

Every instruction, without exception, goes through four stages:

| Stage | What happens |
|---|---|
| **Fetch** | the program counter drives the address; the instruction arrives in the IR |
| **Decode** | the control unit works out what this instruction needs |
| **Execute** | the instruction does its work — arithmetic, a memory access, or a jump |
| **Writeback** | the program counter moves on, unless a jump already moved it |

**What this does.** One call advances exactly one stage — not one instruction. That is
what makes the cycle watchable instead of theoretical.

```cpp
case STAGE_FETCH:      // PC drives the address, instruction loads into IR
case STAGE_DECODE:     // ask the instruction for its control signals
case STAGE_EXECUTE:    // current_->execute(*this)
case STAGE_WRITEBACK:  // PC advances, unless a branch already moved it
```
<sub>the four cases of CPU::step() — src/core/CPU.cpp:128-176, bodies omitted</sub>

The detail worth pointing at: **writeback advances the program counter only if no jump
happened.** A jump works by writing a different number into the PC, so the machine simply
carries on from wherever the PC now points. There is no separate "jumping" machinery — a
jump is just an assignment to one register.

---

## How to explain CO1 in one minute

1. **Architecture answers two questions:** what parts exist, and what can a program ask
   for.
2. **The parts:** a 16-bit word, eight general registers plus PC, IR, SP, MAR and MDR, an
   ALU, memory, and one shared bus connecting them.
3. **The flags are the interesting part** — four facts recorded after each operation.
   `CMP` sets them and `JNZ` reads them, and that pair is what makes a loop possible.
4. **The instruction set** is fourteen instructions with three addressing modes:
   immediate, register and direct.
5. **The cycle** is fetch, decode, execute, writeback — and we can step one *stage* at a
   time, so it can be watched rather than memorised.